# RandomForest pipeline

A Random Forest trains 200 decision trees on random slices of the data and lets them vote. Averaging many trees cancels their individual noise, which is why it handles the non-linear segment boundaries best.

One model, one notebook: the full flow (load → split → SMOTE → `Pipeline` → evaluate) with **RandomForest** as the classifier. Tuned with GridSearchCV in `classification.ipynb`: `n_estimators=200`, `max_depth=20`.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE

df = pd.read_csv("../data/processed/df_clean_clustered.csv")
feats = ["BALANCE", "PURCHASES", "ONEOFF_PURCHASES",
         "INSTALLMENTS_PURCHASES", "CASH_ADVANCE", "CREDIT_LIMIT", "PAYMENTS"]
X = df[feats]
y = df["target"]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)

smote = SMOTE(random_state=0)
print("train before SMOTE:")
print(y_train.value_counts().to_string())
print()

X_train, y_train = smote.fit_resample(X_train, y_train)

print("train after SMOTE:")
print(y_train.value_counts().to_string())


In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=200, max_depth=20, random_state=0)),
])

pipeline.fit(X_train, y_train)


In [ ]:
pred = pipeline.predict(X_test)
print("accuracy: ", round(accuracy_score(y_test, pred), 3))
print("precision:", round(precision_score(y_test, pred, average="weighted"), 3))
print("recall:   ", round(recall_score(y_test, pred, average="weighted"), 3))
print("f1:       ", round(f1_score(y_test, pred, average="weighted"), 3))


In [ ]:
ConfusionMatrixDisplay.from_estimator(
    pipeline, X_test, y_test, cmap="Blues", xticks_rotation=25, colorbar=False)
plt.title("RandomForest")
plt.tight_layout()
plt.show()


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
folds = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="f1_weighted")
print("fold F1:", np.round(folds, 3))
print(f"mean {folds.mean():.3f}   std {folds.std():.3f}")


### Result

Test F1 **≈ 0.97** — the best model of the four, by a clear margin. Expected: the cluster borders learned in FS3 are non-linear, and a forest of trees follows them naturally.

The side-by-side comparison of all four models (confusion matrices, CV boxplots, GridSearch, feature importance) lives in `classification.ipynb`.